In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [22]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.
root2 = math.sqrt(2)
denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 

B0_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]

B1_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]

def random():
    q = QuantumCircuit(1) 
    q.h(0) 
    q.measure_all() 
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    return counts.get("1",0)

def oneThird():
    theta = 2 * math.acos(math.sqrt(1/3))
    q = QuantumCircuit(1)
    q.ry(theta, 0)
    q.measure_all()
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    counts = result.get_counts(compiled)
    return counts.get("1", 0)
    
def randomChoice():
    if oneThird() == 0:
        return 1
    else:
        return 2 + random()

def average(c,n): 
    backend = BasicSimulator()
    compiled = transpile(c, backend)
    job_sim = backend.run(compiled, shots=n)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    count00 = counts.get("00",0) 
    count01 = counts.get("01",0) 
    count10 = counts.get("10",0) 
    count11 = counts.get("11",0) 
    return (count00 - count01 - count10 + count11) / n 

def entangledPair():
    q = QuantumCircuit(2, 2) 
    q.h(0)
    q.cx(0,1)
    q.x(0)
    q.z(1)
    return q

N = 100
runs = int(9*N/2)

alice_operators = []
bob_operators = []
alice_bits = []
bob_bits = []

for i in range(runs):
    Ai = randomChoice()
    Bj = randomChoice()
    alice_operators.append(Ai)
    bob_operators.append(Bj)
    q = entangledPair()

    Ea = randomChoice()   
    if Ea == 1:
        q.h(0)
    elif Ea == 2:
        q.unitary(B0_transform_matrix, [0])
    elif Ea == 3:
        pass
    Eb = randomChoice()   
    if Eb == 1:
        q.h(1)
    elif Eb == 2:
        q.unitary(B0_transform_matrix, [1])
    elif Eb == 3:
        pass
    q.measure(0, 0)
    q.measure(1, 1)
    q.reset(0)
    q.reset(1)
        
    #X
    if Ai == 1:
        q.h(0)
    #W    
    elif Ai == 2:     
        q.unitary(B0_transform_matrix, [0])
    #Z   
    elif Ai == 3:     
        pass
    #W
    if Bj == 1:       
        q.unitary(B0_transform_matrix, [1])
    #Z   
    elif Bj == 2:     
        pass
    #V    
    elif Bj == 3:     
        q.unitary(B1_transform_matrix, [1])

    q.measure(0, 0)
    q.measure(1, 1)
    backend = BasicSimulator()
    compiled = transpile(q, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    results = list(counts.keys())[0]
    alice_bits.append(int(results[1]))
    bob_bits.append(int(results[0]))

shared_key = []

for i in range(runs):
    Ai = alice_operators[i]
    Bj = bob_operators[i]

    if (Ai == 2 and Bj == 1) or (Ai == 3 and Bj == 2):
        bob_bit = 1 - bob_bits[i]
        shared_key.append(alice_bits[i])


circuit_A1_B1 = entangledPair()
Ea = randomChoice()
if Ea == 1:
    circuit_A1_B1.h(0)
elif Ea == 2:
    circuit_A1_B1.unitary(B0_transform_matrix, [0])
elif Ea == 3:
    pass
Eb = randomChoice()
if Eb == 1:
    circuit_A1_B1.h(1)
elif Eb == 2:
    circuit_A1_B1.unitary(B0_transform_matrix, [1])
elif Eb == 3:
    pass
circuit_A1_B1.measure(0, 0)                   
circuit_A1_B1.measure(1, 1)
circuit_A1_B1.reset(0)                         
circuit_A1_B1.reset(1)
circuit_A1_B1.h(0)                              
circuit_A1_B1.unitary(B0_transform_matrix, [1])
circuit_A1_B1.measure(0, 0)                    
circuit_A1_B1.measure(1, 1)
A1_B1_average = average(circuit_A1_B1, N)

circuit_A1_B3 = entangledPair()
Ea = randomChoice()        
if Ea == 1:
    circuit_A1_B3.h(0)
elif Ea == 2:
    circuit_A1_B3.unitary(B0_transform_matrix, [0])
elif Ea == 3:
    pass
Eb = randomChoice()
if Eb == 1:
    circuit_A1_B3.h(1)
elif Eb == 2:
    circuit_A1_B3.unitary(B0_transform_matrix, [1])
elif Eb == 3:
    pass
circuit_A1_B3.measure(0, 0)                    
circuit_A1_B3.measure(1, 1)
circuit_A1_B3.reset(0)                         
circuit_A1_B3.reset(1)
circuit_A1_B3.h(0)                            
circuit_A1_B3.unitary(B1_transform_matrix, [1]) 
circuit_A1_B3.measure(0, 0)                    
circuit_A1_B3.measure(1, 1)
A1_B3_average = average(circuit_A1_B3, N)
 

circuit_A3_B1 = entangledPair()
Ea = randomChoice()       
if Ea == 1:
    circuit_A3_B1.h(0)
elif Ea == 2:
    circuit_A3_B1.unitary(B0_transform_matrix, [0])
elif Ea == 3:
    pass
Eb = randomChoice()
if Eb == 1:
    circuit_A3_B1.h(1)
elif Eb == 2:
    circuit_A3_B1.unitary(B0_transform_matrix, [1])
elif Eb == 3:
    pass
circuit_A3_B1.measure(0, 0)                 
circuit_A3_B1.measure(1, 1)
circuit_A3_B1.reset(0)                         
circuit_A3_B1.reset(1)
circuit_A3_B1.unitary(B0_transform_matrix, [1])
circuit_A3_B1.measure(0, 0)                     
circuit_A3_B1.measure(1, 1)
A3_B1_average = average(circuit_A3_B1, N)
 
circuit_A3_B3 = entangledPair()
Ea = randomChoice()        
if Ea == 1:
    circuit_A3_B3.h(0)
elif Ea == 2:
    circuit_A3_B3.unitary(B0_transform_matrix, [0])
elif Ea == 3:
    pass
Eb = randomChoice()
if Eb == 1:
    circuit_A3_B3.h(1)
elif Eb == 2:
    circuit_A3_B3.unitary(B0_transform_matrix, [1])
elif Eb == 3:
    pass
circuit_A3_B3.measure(0, 0)                   
circuit_A3_B3.measure(1, 1)
circuit_A3_B3.reset(0)                          
circuit_A3_B3.reset(1)
circuit_A3_B3.unitary(B1_transform_matrix, [1])
circuit_A3_B3.measure(0, 0)                 
circuit_A3_B3.measure(1, 1)
A3_B3_average = average(circuit_A3_B3, N)
 
result = abs(A1_B1_average - A1_B3_average + A3_B1_average + A3_B3_average)

if result < 2:
    print(f"Attacker detected, |S| = {result:.3f}")
else:
    print(f"No attacker detected, |S| = {result:.3f}")
print (f"Shared key: {shared_key}")

Attacker detected, |S| = 1.180
Shared key: [1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1]
